# Long Context Training

## Historical Context

The evolution of context length in LLMs:

- **2017**: Original Transformer - 512 tokens
- **2018**: GPT - 512 tokens
- **2019**: GPT-2 - 1024 tokens
- **2020**: GPT-3 - 2048 tokens
- **2021**: RoPE (Rotary Position Embeddings) - better length extrapolation
- **2022**: ALiBi (Attention with Linear Biases) - train short, infer long
- **2023 Feb**: LLaMA - 2048 tokens with RoPE
- **2023 May**: MPT-7B-StoryWriter - 65K tokens
- **2023 June**: RoPE scaling methods (linear, NTK-aware)
- **2023 Aug**: YaRN (Yet another RoPE extensioN) - improved scaling
- **2023 Sep**: Mistral 7B - 32K tokens (8K training + sliding window)
- **2023 Nov**: GPT-4 Turbo - 128K tokens
- **2024**: Gemini 1.5 - 1M tokens, Claude 3 - 200K tokens
- **2025**: Context windows of 1M+ becoming standard

## Why Long Context Matters

**Use cases**:
- Analyze entire codebases
- Summarize books/research papers
- Long-form conversation memory
- Document QA without retrieval
- Multi-document reasoning

**Challenges**:
- O(n²) attention complexity
- Position embedding extrapolation
- KV cache memory explosion
- Training stability
- "Lost in the middle" phenomenon

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
from typing import Tuple, Optional

## 1. RoPE (Rotary Position Embeddings)

RoPE is the foundation for most modern long-context LLMs

In [ ]:
class RotaryPositionEmbedding:
    """RoPE: Rotary Position Embeddings (Su et al., 2021).
    
    Key advantages:
    - Relative position encoding (attends based on distance)
    - Better extrapolation to longer sequences
    - No learned parameters
    - Used by LLaMA, Mistral, GPT-NeoX, etc.
    """
    
    def __init__(self, dim: int, max_seq_len: int = 2048, base: float = 10000.0):
        self.dim = dim
        self.max_seq_len = max_seq_len
        self.base = base
        
        # Precompute frequency tensor
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
        
        # Cache cos/sin for efficiency
        self._cache_cos_sin(max_seq_len)
    
    def _cache_cos_sin(self, seq_len: int):
        """Precompute cos and sin values."""
        t = torch.arange(seq_len).type_as(self.inv_freq)
        freqs = torch.outer(t, self.inv_freq)  # [seq_len, dim//2]
        emb = torch.cat([freqs, freqs], dim=-1)  # [seq_len, dim]
        
        self.cos_cached = emb.cos()
        self.sin_cached = emb.sin()
    
    def rotate_half(self, x: torch.Tensor) -> torch.Tensor:
        """Rotate half the hidden dims of the input."""
        x1, x2 = x.chunk(2, dim=-1)
        return torch.cat([-x2, x1], dim=-1)
    
    def apply_rotary_pos_emb(self, q: torch.Tensor, k: torch.Tensor, 
                            position_ids: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Apply rotary position embeddings to queries and keys.
        
        Args:
            q: [batch, seq_len, num_heads, head_dim]
            k: [batch, seq_len, num_heads, head_dim]
            position_ids: [batch, seq_len]
        """
        # Get cos/sin for these positions
        cos = self.cos_cached[position_ids]
        sin = self.sin_cached[position_ids]
        
        # Apply rotation
        q_embed = (q * cos) + (self.rotate_half(q) * sin)
        k_embed = (k * cos) + (self.rotate_half(k) * sin)
        
        return q_embed, k_embed


# Visualize RoPE frequencies
def visualize_rope_frequencies(dim: int = 64, max_seq_len: int = 2048):
    """Visualize how RoPE encodes position."""
    
    rope = RotaryPositionEmbedding(dim, max_seq_len)
    
    # Get frequencies for all positions
    positions = torch.arange(max_seq_len)
    freqs = torch.outer(positions, rope.inv_freq)
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    # Plot 1: Frequency spectrum
    ax = axes[0, 0]
    ax.plot(rope.inv_freq.numpy())
    ax.set_xlabel('Dimension', fontsize=12)
    ax.set_ylabel('Inverse Frequency', fontsize=12)
    ax.set_title('RoPE Frequency Spectrum', fontsize=14, fontweight='bold')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Position encoding heatmap
    ax = axes[0, 1]
    im = ax.imshow(freqs[:200, :16].numpy(), aspect='auto', cmap='RdBu', 
                   interpolation='nearest')
    ax.set_xlabel('Dimension', fontsize=12)
    ax.set_ylabel('Position', fontsize=12)
    ax.set_title('RoPE Encoding (first 200 positions)', fontsize=14, fontweight='bold')
    plt.colorbar(im, ax=ax)
    
    # Plot 3: Cos/Sin patterns
    ax = axes[1, 0]
    positions_sample = torch.arange(0, 200)
    for i in [0, 4, 8, 12]:
        cos_vals = rope.cos_cached[positions_sample, i].numpy()
        ax.plot(positions_sample, cos_vals, label=f'dim={i}')
    ax.set_xlabel('Position', fontsize=12)
    ax.set_ylabel('Cosine Value', fontsize=12)
    ax.set_title('RoPE Cosine Patterns', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 4: Wavelengths
    ax = axes[1, 1]
    wavelengths = 2 * np.pi / rope.inv_freq.numpy()
    ax.plot(wavelengths)
    ax.set_xlabel('Dimension', fontsize=12)
    ax.set_ylabel('Wavelength (positions)', fontsize=12)
    ax.set_title('RoPE Wavelengths', fontsize=14, fontweight='bold')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('rope_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("RoPE Properties:")
    print(f"  Min wavelength: {wavelengths.min():.1f} positions")
    print(f"  Max wavelength: {wavelengths.max():.1f} positions")
    print(f"  Covers {wavelengths.max() / (2*np.pi):.0f}x the sequence length")

visualize_rope_frequencies()

## 2. RoPE Scaling Methods

How to extend RoPE beyond training length

In [ ]:
class LinearRoPEScaling(RotaryPositionEmbedding):
    """Linear RoPE scaling (simple but works).
    
    Idea: Scale position indices by a factor before computing RoPE.
    If trained on 2K, scale=4 gives 8K context.
    
    Problem: All frequencies scaled equally, loses some information.
    """
    
    def __init__(self, dim: int, max_seq_len: int, base: float = 10000.0, scale: float = 1.0):
        self.scale = scale
        super().__init__(dim, max_seq_len, base)
    
    def _cache_cos_sin(self, seq_len: int):
        """Scale positions linearly."""
        t = torch.arange(seq_len).type_as(self.inv_freq) / self.scale
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        
        self.cos_cached = emb.cos()
        self.sin_cached = emb.sin()


class NTKAwareRoPEScaling(RotaryPositionEmbedding):
    """NTK-aware RoPE scaling (better than linear).
    
    Idea: Scale the base frequency instead of positions.
    Preserves high-frequency information better.
    
    Used by many models extending context length.
    """
    
    def __init__(self, dim: int, max_seq_len: int, base: float = 10000.0, 
                 scale: float = 1.0, original_max_seq_len: int = 2048):
        # Adjust base frequency
        adjusted_base = base * (scale ** (dim / (dim - 2)))
        super().__init__(dim, max_seq_len, adjusted_base)


class YaRNScaling(RotaryPositionEmbedding):
    """YaRN: Yet another RoPE extensioN (Peng et al., 2023).
    
    Most sophisticated scaling method:
    - Uses different scaling for different frequency ranges
    - "Attention temperature" to adjust focus
    - Ramp function for smooth interpolation
    
    State-of-the-art for extending context length.
    """
    
    def __init__(self, dim: int, max_seq_len: int, base: float = 10000.0,
                 scale: float = 1.0, original_max_seq_len: int = 2048,
                 alpha: float = 1.0, beta: float = 32.0):
        self.scale = scale
        self.original_max_seq_len = original_max_seq_len
        self.alpha = alpha  # Low-frequency scaling
        self.beta = beta    # High-frequency threshold
        super().__init__(dim, max_seq_len, base)
    
    def _get_yarn_ramp(self, dim_idx: int, total_dims: int) -> float:
        """Compute ramp function for smooth interpolation."""
        # Low frequencies: scale more (alpha)
        # High frequencies: scale less (1.0)
        ratio = dim_idx / total_dims
        if ratio < self.beta / total_dims:
            return 1.0  # High freq, no scaling
        elif ratio > 1.0 - (self.beta / total_dims):
            return self.alpha  # Low freq, full scaling
        else:
            # Smooth interpolation
            t = (ratio - self.beta / total_dims) / (1.0 - 2 * self.beta / total_dims)
            return 1.0 + (self.alpha - 1.0) * t
    
    def _cache_cos_sin(self, seq_len: int):
        """Apply YaRN scaling."""
        # Compute per-dimension scaling
        dim_scales = torch.tensor([
            self._get_yarn_ramp(i, self.dim // 2)
            for i in range(self.dim // 2)
        ])
        
        # Scale inverse frequencies
        scaled_inv_freq = self.inv_freq / dim_scales
        
        # Compute embeddings
        t = torch.arange(seq_len).type_as(scaled_inv_freq)
        freqs = torch.outer(t, scaled_inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        
        self.cos_cached = emb.cos()
        self.sin_cached = emb.sin()


# Compare scaling methods
def compare_rope_scaling(dim: int = 64, original_len: int = 2048, target_len: int = 8192):
    """Compare different RoPE scaling approaches."""
    
    scale_factor = target_len / original_len
    
    methods = {
        'Original': RotaryPositionEmbedding(dim, original_len),
        'Linear': LinearRoPEScaling(dim, target_len, scale=scale_factor),
        'NTK-Aware': NTKAwareRoPEScaling(dim, target_len, scale=scale_factor, 
                                          original_max_seq_len=original_len),
        'YaRN': YaRNScaling(dim, target_len, scale=scale_factor,
                           original_max_seq_len=original_len, alpha=1.0, beta=32),
    }
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()
    
    for idx, (name, rope) in enumerate(methods.items()):
        ax = axes[idx]
        
        # Sample positions from extended range
        if name == 'Original':
            positions = torch.arange(0, original_len, 10)
        else:
            positions = torch.arange(0, target_len, 40)
        
        # Plot cos patterns for first few dimensions
        for dim_idx in [0, 8, 16, 24]:
            if dim_idx < rope.cos_cached.shape[1]:
                cos_vals = rope.cos_cached[positions, dim_idx].numpy()
                ax.plot(positions, cos_vals, label=f'dim={dim_idx}', alpha=0.7)
        
        ax.set_xlabel('Position', fontsize=11)
        ax.set_ylabel('Cosine Value', fontsize=11)
        ax.set_title(f'{name} Scaling', fontsize=13, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('rope_scaling_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("RoPE Scaling Comparison")
    print("="*80)
    print(f"Original length: {original_len}")
    print(f"Target length: {target_len}")
    print(f"Scale factor: {scale_factor}x")
    print()
    print("Method characteristics:")
    print("  Linear: Simple, but loses high-frequency info")
    print("  NTK-Aware: Better preservation of frequencies")
    print("  YaRN: State-of-the-art, adaptive scaling per frequency")

compare_rope_scaling()

## 3. ALiBi (Attention with Linear Biases)

Alternative to RoPE: train short, test long

In [ ]:
class ALiBiAttention(nn.Module):
    """ALiBi: Attention with Linear Biases (Press et al., 2022).
    
    Key idea: Add position-dependent bias to attention scores.
    No position embeddings needed!
    
    Advantages:
    - Train on short sequences (1K-2K)
    - Infer on much longer (tested up to 1M tokens)
    - No extra parameters
    - Simple and effective
    
    Used by: MPT, BLOOM, some research models
    """
    
    def __init__(self, num_heads: int, max_seq_len: int = 2048):
        super().__init__()
        self.num_heads = num_heads
        self.max_seq_len = max_seq_len
        
        # Compute slopes for each head
        slopes = self._get_slopes(num_heads)
        self.register_buffer('slopes', slopes)
        
        # Precompute bias matrix
        self._build_bias(max_seq_len)
    
    def _get_slopes(self, num_heads: int) -> torch.Tensor:
        """Compute geometric sequence of slopes.
        
        Slopes decrease geometrically: head i gets slope 2^(-8i/n)
        """
        def get_slopes_power_of_2(n):
            start = 2 ** (-2 ** -(math.log2(n) - 3))
            ratio = start
            return [start * (ratio ** i) for i in range(n)]
        
        if math.log2(num_heads).is_integer():
            return torch.tensor(get_slopes_power_of_2(num_heads))
        else:
            # Closest power of 2
            closest_power = 2 ** math.floor(math.log2(num_heads))
            slopes = get_slopes_power_of_2(closest_power)
            slopes += get_slopes_power_of_2(2 * closest_power)[::2][:num_heads - closest_power]
            return torch.tensor(slopes)
    
    def _build_bias(self, seq_len: int):
        """Build bias matrix: distance between positions."""
        # Create position indices
        position_ids = torch.arange(seq_len)
        # Compute relative distances
        relative_positions = position_ids[None, :] - position_ids[:, None]
        relative_positions = relative_positions.abs()  # |i - j|
        
        # Apply slopes: [num_heads, seq_len, seq_len]
        alibi_bias = -relative_positions.unsqueeze(0) * self.slopes.unsqueeze(-1).unsqueeze(-1)
        self.register_buffer('alibi_bias', alibi_bias)
    
    def forward(self, attention_scores: torch.Tensor) -> torch.Tensor:
        """Add ALiBi bias to attention scores.
        
        Args:
            attention_scores: [batch, num_heads, seq_len, seq_len]
        
        Returns:
            Biased attention scores
        """
        seq_len = attention_scores.size(-1)
        
        # Get bias for this sequence length
        bias = self.alibi_bias[:, :seq_len, :seq_len]
        
        # Add bias to attention scores
        return attention_scores + bias


# Visualize ALiBi
def visualize_alibi(num_heads: int = 8, seq_len: int = 64):
    """Visualize ALiBi attention biases."""
    
    alibi = ALiBiAttention(num_heads, seq_len)
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    
    for head_idx in range(num_heads):
        ax = axes[head_idx]
        
        # Get bias for this head
        bias = alibi.alibi_bias[head_idx].numpy()
        slope = alibi.slopes[head_idx].item()
        
        # Plot heatmap
        im = ax.imshow(bias, cmap='RdBu_r', aspect='auto')
        ax.set_xlabel('Key Position', fontsize=10)
        ax.set_ylabel('Query Position', fontsize=10)
        ax.set_title(f'Head {head_idx} (slope={slope:.4f})', fontsize=11, fontweight='bold')
        plt.colorbar(im, ax=ax)
    
    plt.tight_layout()
    plt.savefig('alibi_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Plot slopes
    plt.figure(figsize=(10, 6))
    plt.plot(range(num_heads), alibi.slopes.numpy(), marker='o', linewidth=2, markersize=8)
    plt.xlabel('Head Index', fontsize=12)
    plt.ylabel('Slope (negative weight)', fontsize=12)
    plt.title('ALiBi Slopes per Attention Head', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.yscale('log')
    plt.tight_layout()
    plt.savefig('alibi_slopes.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("ALiBi Properties:")
    print(f"  Number of heads: {num_heads}")
    print(f"  Slopes range: {alibi.slopes.min():.6f} to {alibi.slopes.max():.6f}")
    print(f"  Different heads penalize distance differently")
    print(f"  Some heads focus locally, others globally")

visualize_alibi()

## 4. Memory Optimization for Long Sequences

KV cache grows linearly with sequence length

In [ ]:
def analyze_memory_usage(seq_lengths: list, model_config: dict):
    """Analyze memory requirements for different context lengths."""
    
    batch_size = 1
    hidden_size = model_config['hidden_size']
    num_layers = model_config['num_layers']
    num_heads = model_config['num_heads']
    head_dim = hidden_size // num_heads
    
    results = []
    
    for seq_len in seq_lengths:
        # KV cache size per layer: 2 (K+V) * batch * seq_len * num_heads * head_dim
        # Each element is 2 bytes (fp16) or 4 bytes (fp32)
        kv_cache_per_layer_fp16 = 2 * batch_size * seq_len * num_heads * head_dim * 2
        total_kv_cache_fp16 = kv_cache_per_layer_fp16 * num_layers
        
        # Activations (rough estimate)
        activations = batch_size * seq_len * hidden_size * 4  # Intermediate states
        
        # Total memory (GB)
        total_memory_gb = (total_kv_cache_fp16 + activations) / (1024 ** 3)
        
        results.append({
            'seq_len': seq_len,
            'kv_cache_gb': total_kv_cache_fp16 / (1024 ** 3),
            'total_gb': total_memory_gb,
        })
    
    return results


# Analyze for LLaMA-7B-like model
llama_config = {
    'hidden_size': 4096,
    'num_layers': 32,
    'num_heads': 32,
}

seq_lengths = [2048, 4096, 8192, 16384, 32768, 65536, 131072]
memory_results = analyze_memory_usage(seq_lengths, llama_config)

print("Memory Usage Analysis (LLaMA-7B-like, batch_size=1, fp16)")
print("="*80)
print(f"{'Seq Length':>12} | {'KV Cache (GB)':>15} | {'Total (GB)':>12}")
print("-"*80)
for result in memory_results:
    print(f"{result['seq_len']:>12,} | {result['kv_cache_gb']:>15.2f} | {result['total_gb']:>12.2f}")

# Plot memory growth
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

seq_lens = [r['seq_len'] for r in memory_results]
kv_cache = [r['kv_cache_gb'] for r in memory_results]
total_mem = [r['total_gb'] for r in memory_results]

# Linear scale
ax1.plot(seq_lens, kv_cache, marker='o', linewidth=2, markersize=8, label='KV Cache')
ax1.plot(seq_lens, total_mem, marker='s', linewidth=2, markersize=8, label='Total Memory')
ax1.set_xlabel('Sequence Length', fontsize=12)
ax1.set_ylabel('Memory (GB)', fontsize=12)
ax1.set_title('Memory Growth (Linear Scale)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Log scale
ax2.plot(seq_lens, kv_cache, marker='o', linewidth=2, markersize=8, label='KV Cache')
ax2.plot(seq_lens, total_mem, marker='s', linewidth=2, markersize=8, label='Total Memory')
ax2.set_xlabel('Sequence Length', fontsize=12)
ax2.set_ylabel('Memory (GB)', fontsize=12)
ax2.set_title('Memory Growth (Log Scale)', fontsize=14, fontweight='bold')
ax2.set_xscale('log', base=2)
ax2.set_yscale('log')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('long_context_memory.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey Observations:")
print(f"  - Memory grows linearly with sequence length")
print(f"  - 128K context needs ~{memory_results[-1]['total_gb']:.1f}GB (vs ~{memory_results[0]['total_gb']:.1f}GB for 2K)")
print(f"  - Batch size multiplies memory requirements")
print(f"  - Need optimization for long contexts!")

## 5. Training Strategies for Long Context

In [ ]:
class LongContextTrainingStrategy:
    """Strategies for training with long contexts."""
    
    @staticmethod
    def progressive_length_training(base_length: int = 2048, 
                                   target_length: int = 32768,
                                   num_stages: int = 4):
        """Gradually increase sequence length during training.
        
        Strategy:
        1. Start with base length (2K)
        2. Train for N steps
        3. Double length, train more
        4. Repeat until target length
        
        Benefits:
        - More stable training
        - Faster initial training
        - Better convergence
        """
        stages = []
        current_length = base_length
        
        for stage in range(num_stages):
            # Exponential growth
            length = min(current_length * (2 ** stage), target_length)
            # More steps for longer sequences
            steps = 1000 * (2 ** stage)
            stages.append({
                'stage': stage + 1,
                'length': length,
                'steps': steps,
                'description': f'Train on {length:,} tokens for {steps:,} steps'
            })
            
            if length >= target_length:
                break
        
        return stages
    
    @staticmethod
    def sliding_window_attention(seq_len: int = 32768, window_size: int = 4096):
        """Sliding window attention (used by Mistral).
        
        Each token attends to:
        - Previous window_size tokens
        - Reduces memory from O(n²) to O(n*w)
        
        Benefits:
        - Much less memory
        - Still captures local context
        - Can be combined with global attention on special tokens
        """
        # Memory saved
        full_attention_memory = seq_len ** 2
        sliding_window_memory = seq_len * window_size
        memory_ratio = sliding_window_memory / full_attention_memory
        
        return {
            'seq_len': seq_len,
            'window_size': window_size,
            'memory_reduction': f'{(1 - memory_ratio) * 100:.1f}%',
            'description': f'Each token attends to {window_size:,} previous tokens',
        }
    
    @staticmethod
    def sparse_attention_patterns():
        """Different sparse attention patterns."""
        patterns = {
            'Sliding Window': 'Attend to fixed window of previous tokens',
            'Strided': 'Attend to every k-th token',
            'Fixed': 'All tokens attend to first few tokens',
            'Random': 'Random subset of tokens (Longformer)',
            'Block-Sparse': 'Divide into blocks, attend within/between blocks',
            'Dilated': 'Exponentially increasing gaps',
        }
        return patterns


# Demonstrate training strategies
strategy = LongContextTrainingStrategy()

print("Long Context Training Strategies")
print("="*80)

# Progressive length training
print("\n1. PROGRESSIVE LENGTH TRAINING")
print("-"*80)
stages = strategy.progressive_length_training(2048, 32768, 5)
for stage in stages:
    print(f"Stage {stage['stage']}: {stage['description']}")

# Sliding window
print("\n2. SLIDING WINDOW ATTENTION")
print("-"*80)
window_info = strategy.sliding_window_attention(32768, 4096)
for key, value in window_info.items():
    print(f"{key}: {value}")

# Sparse patterns
print("\n3. SPARSE ATTENTION PATTERNS")
print("-"*80)
patterns = strategy.sparse_attention_patterns()
for name, desc in patterns.items():
    print(f"{name:20s}: {desc}")

# Visualize progressive training
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Training schedule
stage_nums = [s['stage'] for s in stages]
lengths = [s['length'] for s in stages]
steps = [s['steps'] for s in stages]

ax1.plot(stage_nums, lengths, marker='o', linewidth=2, markersize=10, color='blue')
ax1.set_xlabel('Training Stage', fontsize=12)
ax1.set_ylabel('Sequence Length', fontsize=12)
ax1.set_title('Progressive Length Training', fontsize=14, fontweight='bold')
ax1.set_yscale('log', base=2)
ax1.grid(True, alpha=0.3)
for i, length in enumerate(lengths):
    ax1.annotate(f'{length:,}', xy=(stage_nums[i], length), 
                xytext=(5, 5), textcoords='offset points', fontsize=9)

# Training steps
ax2.bar(stage_nums, steps, edgecolor='black', alpha=0.7, color='green')
ax2.set_xlabel('Training Stage', fontsize=12)
ax2.set_ylabel('Number of Steps', fontsize=12)
ax2.set_title('Steps per Stage', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')
for i, step in enumerate(steps):
    ax2.text(stage_nums[i], step, f'{step:,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('progressive_training.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Takeaways

### Context Length Evolution
- **2017-2020**: 512-2048 tokens was standard
- **2023**: 8K-32K becoming common
- **2024-2025**: 100K-1M+ tokens now possible

### Position Encoding Methods

**RoPE (Rotary Position Embeddings)**:
- Most popular for modern LLMs
- Relative position encoding
- Can be extended with scaling methods

**ALiBi (Attention with Linear Biases)**:
- Train short, test long
- No position embeddings
- Simple and effective

### Scaling Methods

1. **Linear Scaling**: Simple but loses information
2. **NTK-Aware**: Better frequency preservation
3. **YaRN**: State-of-the-art, adaptive per frequency

### Training Strategies

**Progressive Length Training**:
- Start short (2K)
- Gradually increase to target
- More stable and efficient

**Sliding Window Attention**:
- Reduces memory from O(n²) to O(n*w)
- Used by Mistral 7B
- Still captures local context

**Sparse Attention**:
- Various patterns (strided, block, dilated)
- Trade-off: memory vs. coverage

### Memory Challenges

**KV Cache**:
- Grows linearly with sequence length
- Dominates memory at long contexts
- 128K context needs ~40GB for 7B model

**Solutions**:
- Quantization (int8/int4 KV cache)
- Flash Attention (memory-efficient attention)
- Paged Attention (vLLM)
- Sliding window

### Best Practices

1. **Start with RoPE + scaling** (most models use this)
2. **Use progressive training** (don't jump to max length)
3. **Monitor memory carefully** (KV cache dominates)
4. **Consider sliding window** if full attention too expensive
5. **Test extrapolation** before claiming long context
6. **Watch for "lost in the middle"** problem

### Production Recommendations

**For 7B model extending 2K → 32K**:
1. Use YaRN scaling (best results)
2. Progressive training: 2K → 4K → 8K → 16K → 32K
3. ~10K steps at each stage
4. Lower LR than original training (1e-5)
5. Evaluate on retrieval tasks (needle in haystack)

**Memory budget**:
- 32K: ~10GB KV cache
- 64K: ~20GB KV cache  
- 128K: ~40GB KV cache
- Plan GPU memory accordingly!